In [1]:
import h5py
import finesse
import networkx as nx
import torch
from model.power_predictor import LinGNN as PowerGNN
from train_power_predictor import PowerDataset

In [2]:
kat = """
# Add a Laser named L0 with a power of 1 W.
l L0 P=1

s s1 portA=L0.p1 portB=eom1.p1 L=10

modulator eom1 9M 0.1 order=1

s s2 portA=eom1.p2 portB=ITM.p1 L=10

# Input mirror of cavity.
m ITM L=0 T=0.014 Rc=-1934

# Intra-cavity space with length of 4 km
s CAV ITM.p2 ETM.p1 L=3994.47

# End mirror of cavity.
m ETM L=0 T=5u  Rc=2245

cavity cavArm source=ITM.p2.o

# Power detectors on reflection, circulation and transmission.
pd circ ETM.p1.i

pd1 pdhI node=ITM.p1.o f=eom1.f phase=0 # In phase demodulated signal
pd1 pdhQ node=ITM.p1.o f=eom1.f phase=90 # Quadrature phase demodulated signal

# dof ETMz ETM.dofs.z
# readout_rf pdh_readout ITM.p1.o f=eom1.f output_detectors=true phase=0

# Add a lock
lock lock_length pdhI ETM.phi -1.0673950644453318 1e-12
"""

In [3]:
def reset_model(kat):
    fabry_perot = finesse.Model()
    fabry_perot.parse(kat)
    fabry_perot.modes(maxtem=6, modes='even')
    return fabry_perot

In [4]:
def model_to_nx_port(model):

    finesse_g = model.optical_network
    g = nx.DiGraph()
    
    for node in finesse_g.nodes():
        print(f"Processing node: {node}")
        name = node.replace('.', '_')
        opt = node.split('.')[0]
        
        if isinstance(getattr(model, opt), finesse.components.mirror.Mirror):
            # Create feature vector
            print("add a mirror node")
            g.add_node(node, Rc=getattr(model, opt).Rcx.value, R = getattr(model, opt).R.value, alpha=0)
        elif isinstance(getattr(model, opt), finesse.components.laser.Laser):
            print("add a laser node")
            g.add_node(node, Rc=0, R = 0, alpha=0)
        # elif isinstance(getattr(model, opt), finesse.components.beamsplitter.Beamsplitter):
        #     g.add_node(node, Rc=getattr(model, opt).Rcx.value, R = getattr(model, opt).R.value, alpha = getattr(model, opt).alpha.value)
        # elif isinstance(getattr(model, opt), finesse.components.lens.Lens):
        #     g.add_node(node, Rc=2*getattr(model, opt).f.value, R = 0, alpha = 0)
    # Access edge attributes
    for i, edge in enumerate(finesse_g.edges().data()):
        print(f"Processing edge: {edge}")
        dat = list(finesse_g.edges().data())[i][2]['owner']()
        if not isinstance(dat, finesse.components.space.Space):
            print("add a space edge")
            g.add_edge(str(edge[0]), str(edge[1]), length=0, nr=1)
        else:
            print("add an unknown edge")
            g.add_edge(str(edge[0]), str(edge[1]), length=dat.L.value if not isinstance(dat.L.value, finesse.symbols.Symbol) else dat.L.value.eval(), nr=dat.nr.value if not hasattr(dat.nr.value, 'eval') else dat.nr.value.eval())
    
    return g

In [7]:
finesse_model = reset_model(kat)
# finesse.tb()
result = model_to_nx_port(finesse_model)
result.nodes

Processing node: L0.p1.i
add a laser node
Processing node: L0.p1.o
add a laser node
Processing node: eom1.p1.i
Processing node: eom1.p1.o
Processing node: eom1.p2.i
Processing node: eom1.p2.o
Processing node: ITM.p1.i
add a mirror node
Processing node: ITM.p1.o
add a mirror node
Processing node: ITM.p2.i
add a mirror node
Processing node: ITM.p2.o
add a mirror node
Processing node: ETM.p1.i
add a mirror node
Processing node: ETM.p1.o
add a mirror node
Processing node: ETM.p2.i
add a mirror node
Processing node: ETM.p2.o
add a mirror node
Processing edge: ('L0.p1.o', 'eom1.p1.i', {'name': 'P1i_P2o', 'in_ref': <weakref at 0x7f81e0309530; to 'OpticalNode' at 0x7f81e0315520>, 'out_ref': <weakref at 0x7f81e0309b20; to 'OpticalNode' at 0x7f81e0316fc0>, 'owner': <weakref at 0x7f81e0309d00; to 'Space' at 0x7f81e0317770>, 'length': 1, 'coupling_type': <CouplingType.OPTICAL_TO_OPTICAL: 0>, 'internal': False})
add an unknown edge
Processing edge: ('eom1.p1.i', 'eom1.p2.o', {'name': 'P1i_P2o', 'in

NodeView(('L0.p1.i', 'L0.p1.o', 'ITM.p1.i', 'ITM.p1.o', 'ITM.p2.i', 'ITM.p2.o', 'ETM.p1.i', 'ETM.p1.o', 'ETM.p2.i', 'ETM.p2.o', 'eom1.p1.i', 'eom1.p2.o', 'eom1.p1.o', 'eom1.p2.i'))

In [12]:
model = PowerGNN(hidden_size=700, num_layers=20, lin_layers=6, target_size = 1)
model.load_state_dict(torch.load(f'results/power_train_results/power_aligo_pretrained_fp.pt', map_location=torch.device('cpu')))

/tmp/ipykernel_2060780/3744907882.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'results/power_train_results/power_aligo_pretrained_f

RuntimeError: Error(s) in loading state_dict for LinGNN:
	Missing key(s) in state_dict: "bnn.module.weight", "bnn.module.bias", "bnn.module.running_mean", "bnn.module.running_var", "bne.module.weight", "bne.module.bias", "bne.module.running_mean", "bne.module.running_var", "batch_norms.0.module.weight", "batch_norms.0.module.bias", "batch_norms.0.module.running_mean", "batch_norms.0.module.running_var", "batch_norms.1.module.weight", "batch_norms.1.module.bias", "batch_norms.1.module.running_mean", "batch_norms.1.module.running_var", "batch_norms.2.module.weight", "batch_norms.2.module.bias", "batch_norms.2.module.running_mean", "batch_norms.2.module.running_var", "batch_norms.3.module.weight", "batch_norms.3.module.bias", "batch_norms.3.module.running_mean", "batch_norms.3.module.running_var", "batch_norms.4.module.weight", "batch_norms.4.module.bias", "batch_norms.4.module.running_mean", "batch_norms.4.module.running_var", "batch_norms.5.module.weight", "batch_norms.5.module.bias", "

In [7]:
dataset = PowerDataset(data_files=['data/half_aligo_many_tops.h5'])

29999


In [15]:
dataset_fp = PowerDataset(data_files=['data/fabry_perot_data.h5'])

30000
